# GPU analysis

Purpose: compare CUDA execution across consumer GPUs and distinguish full GPU offload, partial layer offload, and MoE CPU-expert offload.

This notebook reads only `results/processed/final-analysis-dataset.csv` and writes derived figures and tables under `analysis/`. Missing measurements are excluded from the relevant calculation rather than replaced with zero.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import load_dataset, save_figure, save_table, successful, grouped_bar, add_display_hardware

data = load_dataset(PROJECT_ROOT / "results" / "processed" / "final-analysis-dataset.csv")
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")

## GPU analysis subset

Only CUDA rows are included. Successful executions are used for speed, memory, and power summaries; offload categories remain explicit in all summary tables.

In [ ]:
gpu = add_display_hardware(data.loc[data["backend"].eq("cuda")].copy())
gpu_success = successful(gpu)
gpu_summary = gpu_success.groupby(["hardware_label", "model", "experiment_category"], as_index=False).agg(decode_tps=("decode_tps", "mean"), prompt_eval_tps=("prompt_eval_tps", "mean"), vram_usage=("vram_usage", "mean"), gpu_power=("gpu_power", "mean"), model_size_b=("model_size_b", "first"))
display(gpu_summary)
save_table(gpu_summary, "03_gpu_performance_summary.csv")
category_summary = gpu_success.groupby("experiment_category", as_index=False).agg(executions=("experiment_id", "size"), hardware=("hardware", "nunique"), models=("model", "nunique"))
display(category_summary)
save_table(category_summary, "03_gpu_offload_category_summary.csv")

## Decode throughput by GPU

The first chart compares mean generation throughput for each GPU/model pair. The underlying summary includes the offload category so full, partial, and MoE offload can be separated in tables.

In [ ]:
grouped_bar(gpu_success, "hardware_label", "decode_tps", "model", "GPU decode throughput by hardware and model", "Mean decode throughput (tokens/s)", "03_gpu_decode_tps.png", rotate_labels=True)

## Prompt evaluation throughput

Prompt evaluation speed is shown independently because prompt processing and token decoding can respond differently to GPU resources.

In [ ]:
grouped_bar(gpu_success, "hardware_label", "prompt_eval_tps", "model", "GPU prompt evaluation throughput", "Mean prompt evaluation throughput (tokens/s)", "03_gpu_prompt_eval_tps.png", rotate_labels=True)

## VRAM usage versus model size

The scatter plot relates measured VRAM use to nominal model size and uses marker colour for offload category.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for category, subset in gpu_success.dropna(subset=["model_size_b", "vram_usage"]).groupby("experiment_category"):
    ax.scatter(subset["model_size_b"], subset["vram_usage"], label=category, alpha=0.75)
ax.set_title("GPU VRAM usage versus model size")
ax.set_xlabel("Model size (B parameters)")
ax.set_ylabel("Mean VRAM usage (MB)")
ax.legend(title="Offload category")
ax.grid(alpha=0.25)
save_figure(fig, "03_gpu_vram_vs_model_size.png")
plt.show()

## GPU power consumption

Power is compared only where the monitor recorded a value. This avoids treating missing power data as zero consumption.

In [ ]:
grouped_bar(gpu_success.dropna(subset=["gpu_power"]), "hardware_label", "gpu_power", "model", "GPU power consumption", "Mean GPU power (W)", "03_gpu_power.png", rotate_labels=True)

## Performance per watt

Performance per watt is calculated as decode throughput divided by measured GPU power. It is reported only for positive, non-missing power values.

In [ ]:
ppw = gpu_success.loc[gpu_success["gpu_power"].gt(0)].copy()
ppw["decode_tps_per_watt"] = ppw["decode_tps"] / ppw["gpu_power"]
ppw_summary = ppw.groupby(["hardware_label", "model", "experiment_category"], as_index=False).agg(decode_tps_per_watt=("decode_tps_per_watt", "mean"), decode_tps=("decode_tps", "mean"), gpu_power=("gpu_power", "mean"))
display(ppw_summary)
save_table(ppw_summary, "03_gpu_performance_per_watt.csv")
fig, ax = plt.subplots(figsize=(11, 6))
ppw_summary.pivot(index="hardware_label", columns="model", values="decode_tps_per_watt").plot(kind="bar", ax=ax)
ax.set_title("GPU performance per watt")
ax.set_xlabel("GPU")
ax.set_ylabel("Decode throughput per watt (tokens/s/W)")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "03_gpu_performance_per_watt.png")
plt.show()